## B7 · Text data in pandas (.str)

Converting, cleaning, editing, combining, splitting and extracting text in DataFrame columns.

In [589]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats

# Plot defaults: wide 16:9 figures, sharp ("retina") output and slightly smaller text
%config InlineBackend.figure_format = "retina"
plt.rcParams.update({
    "figure.figsize": (6.4, 3.6),
    "figure.dpi": 100,
    "font.size": 9,
    "axes.titlesize": 10,
    "axes.labelsize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 8,
})

# Same data as sns.load_dataset('penguins'), saved locally so it works offline
df_penguins = pd.read_csv("data/penguins.csv")

# A small table of people, with inconsistent text (spaces, capitals), used throughout this sheet
df_people = pd.DataFrame({
    "full_name": ["  Ada Lovelace ", "alan TURING", "Grace Hopper", "Linus Torvalds"],
    "email": ["ada@x.org", "alan@y.com", "grace@x.org", "linus@z.net"],
    "tags": ["a, b", "c", "a, c, d", "b"],
})

### Convert and check

#### Converting a column to string type

**B7.01**

`s.astype("string")`

***(non-examinable)***

The `.astype("string")` method allows us to turn a column into pandas' text type so the `.str` methods can be used. It is useful for numeric IDs or codes that should be treated as text. Alternatively, `.astype(str)` also works but stores the text as the older `object` type.

Before: the IDs are numbers (no quotes around them).

In [590]:
# display the example data
df_ids = pd.DataFrame({"id": [101, 102, 1003]})
df_ids

,id
0,101
1,102
2,1003


In [591]:
df_ids.dtypes

id    int64
dtype: object

In [592]:
df_ids["id"].tolist()

[101, 102, 1003]

After: the same IDs as text (the quotes show they are strings), so `.str` methods now work.

In [593]:
df_ids["id"] = df_ids["id"].astype("string")
df_ids.dtypes

id    string[python]
dtype: object

In [594]:
df_ids["id"].tolist()

['101', '102', '1003']

In [595]:
df_ids["id"].str.len()        # number of characters in each ID

0    3
1    3
2    4
Name: id, dtype: Int64

*See also:* Changing a column's type (B2) · Counting characters in text (B7) · Checking the column types (B1)

#### Counting characters in text

**B7.02**

`s.str.len()`

***(non-examinable)***

The `.str.len()` method allows us to count the characters in every entry of a text column. It is useful for finding entries that are too short or long, or for checking codes have the right length. Alternatively, `.apply(len)` does the same more slowly.

In [596]:
# display the example data
df_people["email"]

0      ada@x.org
1     alan@y.com
2    grace@x.org
3    linus@z.net
Name: email, dtype: object

In [597]:
df_people["email"].str.len()

0     9
1    10
2    11
3    11
Name: email, dtype: int64

*See also:* Applying a function (B9) · Converting a column to string type (B7) · Checking what characters text contains (B7)

#### Checking what characters text contains

**B7.03**

`s.str.isdigit(), .isalpha(), .isupper() ...`

***(non-examinable)***

The `.str.isdigit()`, `.isalpha()` and `.isupper()` methods allow us to test what type of characters each entry contains. They are useful for finding entries that are not numbers, or are in the wrong case. Alternatively, `pd.to_numeric(errors="coerce")` tests whether text can be a number.

In [598]:
# display the example data
col_codes = pd.Series(["123", "abc", "a1", "456"])
col_codes

0    123
1    abc
2     a1
3    456
dtype: object

In [599]:
col_codes.tolist()

['123', 'abc', 'a1', '456']

In [600]:
col_codes.str.isdigit().tolist()

[True, False, False, True]

In [601]:
col_codes.str.isalpha().tolist()

[False, True, False, False]

*See also:* Counting characters in text (B7) · Filtering rows by text (B7) · Building a mask (B3)

### Clean: whitespace and case

#### Stripping whitespace

**B7.04**

`s.str.strip()`

***(non-examinable)***

The `.str.strip()` method allows us to remove spaces from the start and end of each entry. It is useful because invisible spaces stop values matching (`"Ada"` is not `"Ada "`). Alternatively, `.lstrip()` and `.rstrip()` clean only one end.

In [602]:
# display the example data
df_people["full_name"].tolist()

['  Ada Lovelace ', 'alan TURING', 'Grace Hopper', 'Linus Torvalds']

In [603]:
col_cleaned = df_people["full_name"].str.strip()
col_cleaned.tolist()

['Ada Lovelace', 'alan TURING', 'Grace Hopper', 'Linus Torvalds']

In [604]:
print(f'{df_people["full_name"].str.len().tolist()} -> {col_cleaned.str.len().tolist()}')

[15, 11, 12, 14] -> [12, 11, 12, 14]


*See also:* Changing case (B7) · Chaining string methods (B7) · Replacing text (B7)

#### Changing case

**B7.05**

`s.str.lower(), .upper(), .title()`

***(non-examinable)***

The `.str.lower()`, `.upper()` and `.title()` methods allow us to standardise the capital letters in text. They are useful so that `"Adelie"`, `"adelie"` and `"ADELIE"` count as the same category. Alternatively, `.str.casefold()` is a stricter lowercase for comparisons.

In [605]:
# display the example data
df_people["full_name"].tolist()

['  Ada Lovelace ', 'alan TURING', 'Grace Hopper', 'Linus Torvalds']

In [606]:
s = df_people["full_name"].str.strip()
s.tolist()

['Ada Lovelace', 'alan TURING', 'Grace Hopper', 'Linus Torvalds']

In [607]:
s.str.lower().tolist()

['ada lovelace', 'alan turing', 'grace hopper', 'linus torvalds']

In [608]:
s.str.upper().tolist()

['ADA LOVELACE', 'ALAN TURING', 'GRACE HOPPER', 'LINUS TORVALDS']

In [609]:
s.str.title().tolist()

['Ada Lovelace', 'Alan Turing', 'Grace Hopper', 'Linus Torvalds']

*See also:* Stripping whitespace (B7) · Chaining string methods (B7)

#### Chaining string methods

**B7.06**

`s.str.strip().str.lower()`

***(non-examinable)***

Chaining `.str` methods allows us to clean text in several steps on one line. It is useful for the usual "strip, then fix the case" routine. Alternatively, do each step on its own line and reassign, which lets each step be checked on its own.

Each `.str` call must be repeated in the chain.

In [610]:
# display the example data
df_people["full_name"].tolist()

['  Ada Lovelace ', 'alan TURING', 'Grace Hopper', 'Linus Torvalds']

In [611]:
df_people["full_name"].str.strip().str.title()

0      Ada Lovelace
1       Alan Turing
2      Grace Hopper
3    Linus Torvalds
Name: full_name, dtype: object

In [612]:
col_name = df_people["full_name"].str.strip()    # remove spaces
col_name = col_name.str.title()                  # capitalise each word
col_name

0      Ada Lovelace
1       Alan Turing
2      Grace Hopper
3    Linus Torvalds
Name: full_name, dtype: object

*See also:* Stripping whitespace (B7) · Changing case (B7) · Chaining methods over several lines (B4) · Method chaining (B4)

### Edit and combine

#### Replacing text

**B7.07**

`s.str.replace('_', ' ', regex=False)`

***(non-examinable)***

The `.str.replace()` method allows us to swap one piece of text for another in every entry. It is useful for fixing typos, removing characters or tidying column names. Alternatively, `Series.replace({...})` swaps *whole* values using a dictionary rather than parts of text.

`regex=False` treats the pattern as plain text (safer for characters like `.` or `(`).

In [613]:
# display the example data
cols = pd.Series(["body_mass_g", "bill_depth_mm"])
cols

0      body_mass_g
1    bill_depth_mm
dtype: object

In [614]:
cols.tolist()

['body_mass_g', 'bill_depth_mm']

In [615]:
cols.str.replace("_", " ", regex=False)

0      body mass g
1    bill depth mm
dtype: object

*See also:* Lowercasing every column name (B2)

#### Removing a prefix or suffix

**B7.08**

`s.str.removeprefix(), s.str.removesuffix()`

***(non-examinable)***

The `.str.removeprefix()` and `.str.removesuffix()` methods allow us to cut a known start or end off each entry. They are useful for stripping units or domain names. Alternatively, `.str.replace()` can do the same but would also change matches in the middle.

In [616]:
# display the example data
df_people["email"]

0      ada@x.org
1     alan@y.com
2    grace@x.org
3    linus@z.net
Name: email, dtype: object

In [617]:
df_people["email"].str.removesuffix(".org")     # only emails that end in .org change

0          ada@x
1     alan@y.com
2        grace@x
3    linus@z.net
Name: email, dtype: object

*See also:* Replacing part of a string by position (B7)

#### Replacing part of a string by position

**B7.09**

`s.str.slice_replace(start, stop, repl)`

***(non-examinable)***

The `.str.slice_replace()` method allows us to overwrite characters between two positions. It is useful for masking or fixing fixed-width codes. Alternatively, slice and join with `.str[:2] + "XX" + .str[2:]`.

In [618]:
# display the example data
col_codes = pd.Series(["abcdef", "uvwxyz"])
col_codes

0    abcdef
1    uvwxyz
dtype: object

In [619]:
col_codes.str.slice_replace(0, 2, "XX")        # replace characters 0 and 1

0    XXcdef
1    XXwxyz
dtype: object

*See also:* Replacing text (B7) · Removing a prefix or suffix (B7) · Joining text columns (B7)

#### Joining text columns

**B7.10**

`df['a'] + ' ' + df['b']`

***(non-examinable)***

The `+` operator on text columns allows us to join them into one. It is useful for building full names or labels from parts. Alternatively, `.str.cat(other, sep=" ")` joins with a separator in one call.

In [620]:
# display the example data
df = pd.DataFrame({"first": ["Ada", "Alan"], "last": ["Lovelace", "Turing"]})
df

,first,last
0,Ada,Lovelace
1,Alan,Turing


In [621]:
df["full"] = df["first"] + " " + df["last"]
df

,first,last,full
0,Ada,Lovelace,Ada Lovelace
1,Alan,Turing,Alan Turing


*See also:* Joining two lists (A3) · Adding a calculated column (B4) · Converting a column to string type (B7)

### Search and test

#### Testing for contained, starting and ending text

**B7.11**

`s.str.contains(), .startswith(), .endswith()`

***(non-examinable)***

The `.str.contains()`, `.startswith()` and `.endswith()` methods allow us to test each entry for a piece of text, giving True/False. They are useful for building masks to filter on text. Alternatively, `.isin([...])` tests for exact matches against a list.

Watch out: `contains` treats its pattern as a regular expression by default, so `.` matches any character. `regex=False` searches for the exact text.

In [622]:
# display the example data
df_people["email"]

0      ada@x.org
1     alan@y.com
2    grace@x.org
3    linus@z.net
Name: email, dtype: object

In [623]:
df_people['email'].str.contains('x.org', regex=False).tolist()

[True, False, True, False]

In [624]:
df_people['email'].str.startswith('a').tolist()

[True, True, False, False]

In [625]:
df_people['email'].str.endswith('.com').tolist()

[False, True, False, False]

*See also:* Filtering with `~`, `isin()` and `between()` (B3) · Filtering rows by text (B7) · Building a mask (B3)

#### Filtering rows by text

**B7.12**

`df[df['col'].str.contains(pattern)]`

***(non-examinable)***

Using `.str.contains()` as a mask allows us to keep only the rows whose text matches. It is useful for selecting records by a keyword. Alternatively, `isin()` suits exact matches against a list.

In [626]:
# display the example data
df_people

,full_name,email,tags
0,Ada Lovelace,ada@x.org,"a, b"
1,alan TURING,alan@y.com,c
2,Grace Hopper,grace@x.org,"a, c, d"
3,Linus Torvalds,linus@z.net,b


In [627]:
df_people[df_people["email"].str.contains("x.org", regex=False)]

,full_name,email,tags
0,Ada Lovelace,ada@x.org,"a, b"
2,Grace Hopper,grace@x.org,"a, c, d"


*See also:* Filtering with `~`, `isin()` and `between()` (B3) · Testing for contained, starting and ending text (B7) · Filtering rows with a mask (B3)

#### Counting occurrences in text

**B7.13**

`s.str.count(',')`

***(non-examinable)***

The `.str.count()` method allows us to count how many times a piece of text appears in each entry. It is useful for counting items in a comma-separated list. Alternatively, `.str.split(",").str.len()` counts the parts directly.

In [628]:
# display the example data
df_people["tags"]

0       a, b
1          c
2    a, c, d
3          b
Name: tags, dtype: object

In [629]:
df_people["tags"].str.count(",") + 1      # commas + 1 = number of tags

0    2
1    1
2    3
3    1
Name: tags, dtype: int64

*See also:* Counting characters in text (B7) · Splitting into a list (B7) · Testing for contained, starting and ending text (B7) · Filtering rows by text (B7)

### Split and extract

#### Splitting into a list

**B7.14**

`s.str.split(', ')`

***(non-examinable)***

The `.str.split()` method allows us to break each entry into pieces at a separator, giving a list. It is useful for comma-separated values packed into one cell. Alternatively, `expand=True` puts the pieces in separate columns.

In [630]:
# display the example data
df_people["tags"]

0       a, b
1          c
2    a, c, d
3          b
Name: tags, dtype: object

In [631]:
df_people["tags"].str.split(", ")

0       [a, b]
1          [c]
2    [a, c, d]
3          [b]
Name: tags, dtype: object

*See also:* Splitting into columns (B7) · Counting occurrences in text (B7) · Taking one part of a split (B7)

#### Splitting into columns

**B7.15**

`s.str.split(' ', n=1, expand=True)`

***(non-examinable)***

The `expand=True` option allows us to split text into separate columns. It is useful for turning `"Ada Lovelace"` into first and last names. Alternatively, `.str.extract()` with a regular expression picks out chosen parts.

`n=1` splits only at the first space.

In [632]:
# display the example data
df_people["full_name"]

0      Ada Lovelace 
1        alan TURING
2       Grace Hopper
3     Linus Torvalds
Name: full_name, dtype: object

In [633]:
col_names = df_people["full_name"].str.strip()
col_names.tolist()

['Ada Lovelace', 'alan TURING', 'Grace Hopper', 'Linus Torvalds']

In [634]:
df_parts = col_names.str.split(" ", n=1, expand=True)
df_parts.columns = ["first", "last"]
df_parts

,first,last
0,Ada,Lovelace
1,alan,TURING
2,Grace,Hopper
3,Linus,Torvalds


*See also:* Splitting into a list (B7) · Taking one part of a split (B7) · Moving the index back into a column (B2)

#### Taking one part of a split

**B7.16**

`s.str.split(' ', expand=True)[0]`

***(non-examinable)***

Indexing the expanded result with `[0]` allows us to keep only one piece of each split. It is useful for first names or file extensions. Alternatively, `.str.split(...).str[0]` does the same without `expand`.

In [635]:
# display the example data
df_people["full_name"].str.strip()

0      Ada Lovelace
1       alan TURING
2      Grace Hopper
3    Linus Torvalds
Name: full_name, dtype: object

In [636]:
col_names = df_people["full_name"].str.strip()
col_names.str.split(" ", expand=True)[0]

0      Ada
1     alan
2    Grace
3    Linus
Name: 0, dtype: object

*See also:* Splitting into columns (B7) · Splitting into a list (B7) · Splitting from the right (B7)

#### Splitting from the right

**B7.17**

`s.str.rsplit(' ', n=1)`

***(non-examinable)***

The `.str.rsplit()` method allows us to split starting at the right-hand end. It is useful for taking off the last word or piece. Alternatively, `.str.rpartition()` returns the three parts around the last separator.

In [637]:
# display the example data
col_names = pd.Series(["Ada King Lovelace", "Alan Turing"])
col_names

0    Ada King Lovelace
1          Alan Turing
dtype: object

In [638]:
col_names.str.rsplit(" ", n=1)

0    [Ada King, Lovelace]
1          [Alan, Turing]
dtype: object

*See also:* Splitting into columns (B7) · Partitioning around a separator (B7) · Taking one part of a split (B7)

#### Partitioning around a separator

**B7.18**

`s.str.partition(': ')`

***(non-examinable)***

The `.str.partition()` method allows us to split each entry into three columns: before, separator, after. It is useful for `key: value` text. Alternatively, `split(..., n=1, expand=True)` drops the separator column.

In [639]:
# display the example data
col_pairs = pd.Series(["key: value", "k2: v2"])
col_pairs

0    key: value
1        k2: v2
dtype: object

In [640]:
col_pairs.str.partition(": ")

,0,1,2
0,key,:,value
1,k2,:,v2


*See also:* Splitting into columns (B7) · Splitting into a list (B7) · Taking one part of a split (B7)